In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

In [2]:
# Load dataset
df = pd.read_excel("dataset/1500dataset.xlsx")

print(df.shape)
print(df.head())

(1500, 4)
   Question_ID  LLM_name                                          LLM_Input  \
0            1   ChatGPT  Explain why the sky appears blue clearly in 3-...   
1            1    Claude  Explain why the sky appears blue clearly in 3-...   
2            1  DeepSeek  Explain why the sky appears blue clearly in 3-...   
3            2   ChatGPT      Explain ocean tides clearly in 3-5 sentences.   
4            2    Claude      Explain ocean tides clearly in 3-5 sentences.   

                                          LLM_output  
0  Sunlight reaches Earth's atmosphere and contai...  
1  Sunlight contains all colors of visible light....  
2  Sunlight contains all colors of the visible sp...  
3  Ocean tides are the periodic rise and fall of ...  
4  Tides are caused mainly by the Moon's gravity ...  


In [3]:
# Check data
print(df.isnull().sum())
print(df["LLM_name"].value_counts())

Question_ID    0
LLM_name       0
LLM_Input      0
LLM_output     0
dtype: int64
LLM_name
ChatGPT     500
Claude      500
DeepSeek    500
Name: count, dtype: int64


In [4]:
# Create labels
labels = {"ChatGPT": 0, "Claude": 1, "DeepSeek": 2}

df["label"] = df["LLM_name"].map(labels)

print(df[["LLM_name", "label"]].head())

   LLM_name  label
0   ChatGPT      0
1    Claude      1
2  DeepSeek      2
3   ChatGPT      0
4    Claude      1


In [5]:
# Split dataset

# I generate 3 AI answer base on 500 questions
question_ids = df["Question_ID"].unique()

# 70 train; 15 validation; 15 test
train_ids, temp_ids = train_test_split(
    question_ids, test_size=0.3, random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, random_state=42
)

train_df = df[df["Question_ID"].isin(train_ids)]
val_df = df[df["Question_ID"].isin(val_ids)]
test_df = df[df["Question_ID"].isin(test_ids)]

print(len(train_df))
print(len(val_df))
print(len(test_df))

1050
225
225


In [6]:
# Prepare text data   Output Only
train_texts = train_df["LLM_output"].astype(str).tolist()
val_texts = val_df["LLM_output"].astype(str).tolist()
test_texts = test_df["LLM_output"].astype(str).tolist()

train_labels = train_df["label"].tolist()
val_labels = val_df["label"].tolist()
test_labels = test_df["label"].tolist()

In [7]:
# Build vocabulary
word_count = Counter()

for text in train_texts:
    words = text.lower().split()
    word_count.update(words)

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, count in word_count.items():
    if count >= 2:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 7428


In [8]:
# Encode text

#since answer around 150 words
max_len = 200

def encode(text):
    words = text.lower().split()
    numbers = []

    for word in words:
        numbers.append(vocab.get(word, 1))

    numbers = numbers[:max_len]

    while len(numbers) < max_len:
        numbers.append(0)

    return numbers

print(encode(train_texts[0])[:20])

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 5, 16, 17, 10, 5, 18]


In [9]:
# Convert to tensors
X_train = torch.tensor([encode(text) for text in train_texts], dtype=torch.long)
X_val = torch.tensor([encode(text) for text in val_texts], dtype=torch.long)
X_test = torch.tensor([encode(text) for text in test_texts], dtype=torch.long)

y_train = torch.tensor(train_labels, dtype=torch.long)
y_val = torch.tensor(val_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([1050, 200])
torch.Size([225, 200])
torch.Size([225, 200])


In [10]:
# Create DataLoader
train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

33
8
8


In [11]:
# CNN model

# in the first code there occurs overfitting so add dropout layer
class CNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.conv = nn.Conv1d(128, 128, kernel_size=3)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, 3)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = torch.relu(self.conv(x))
        x = self.pool(x).squeeze(2)
        x = self.dropout(x)
        x = self.fc(x)

        return x

# Initialize CNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(len(vocab)).to(device)

print(device)
# make sure my GPU works

print(model)

cuda
CNN(
  (embedding): Embedding(7428, 128, padding_idx=0)
  (conv): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
  (pool): AdaptiveMaxPool1d(output_size=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [12]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

In [13]:
# Train CNN
best_acc = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "cnn_output_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {correct/total:.4f}")

Epoch 1/10, Loss: 1.2771, Val Accuracy: 0.4356
Epoch 2/10, Loss: 0.9937, Val Accuracy: 0.5689
Epoch 3/10, Loss: 0.8579, Val Accuracy: 0.6311
Epoch 4/10, Loss: 0.7411, Val Accuracy: 0.5956
Epoch 5/10, Loss: 0.6416, Val Accuracy: 0.6044
Epoch 6/10, Loss: 0.5680, Val Accuracy: 0.6000
Epoch 7/10, Loss: 0.4605, Val Accuracy: 0.6356
Epoch 8/10, Loss: 0.3741, Val Accuracy: 0.6267
Epoch 9/10, Loss: 0.3380, Val Accuracy: 0.6311
Epoch 10/10, Loss: 0.3062, Val Accuracy: 0.6089


In [14]:
# Test CNN
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        outputs = model(X)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

print("Test Accuracy:", correct / total)

Test Accuracy: 0.6755555555555556


In [15]:
# CNN evaluation
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)

        outputs = model(X)
        predictions = torch.argmax(outputs, dim=1)

        y_true.extend(y.tolist())
        y_pred.extend(predictions.cpu().tolist())

print(classification_report(
    y_true, y_pred,
    target_names=["ChatGPT", "Claude", "DeepSeek"],
    digits=4
))

print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

     ChatGPT     0.6224    0.8133    0.7052        75
      Claude     0.8974    0.4667    0.6140        75
    DeepSeek     0.6364    0.7467    0.6871        75

    accuracy                         0.6756       225
   macro avg     0.7187    0.6756    0.6688       225
weighted avg     0.7187    0.6756    0.6688       225

[[61  2 12]
 [20 35 20]
 [17  2 56]]


In [16]:
# Load best CNN
model.load_state_dict(torch.load(
    "cnn_output_best.pth",
    map_location=device,
    weights_only=True
))

model.eval()

CNN(
  (embedding): Embedding(7428, 128, padding_idx=0)
  (conv): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
  (pool): AdaptiveMaxPool1d(output_size=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

In [17]:
# Test CNN
y_true = []
y_pred = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)

        outputs = model(X)
        predictions = torch.argmax(outputs, dim=1)

        y_true.extend(y.tolist())
        y_pred.extend(predictions.cpu().tolist())

print("Test Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

Test Accuracy: 0.6888888888888889
Macro F1: 0.6896256471525288


In [18]:
# Save CNN results
cnn_acc = accuracy_score(y_true, y_pred)
cnn_f1 = f1_score(y_true, y_pred, average="macro")

print("CNN Accuracy:", cnn_acc)
print("CNN Macro F1:", cnn_f1)

CNN Accuracy: 0.6888888888888889
CNN Macro F1: 0.6896256471525288


In [19]:
# LSTM model
class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, 3)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()
        x = self.embedding(x)
        x = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        _, (h, _) = self.lstm(x)
        x = self.dropout(h[-1])
        x = self.fc(x)

        return x

In [20]:
# Initialize LSTM
lstm_model = LSTM(len(vocab)).to(device)

print(lstm_model)

# Loss and optimizer
lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

epochs = 10

LSTM(
  (embedding): Embedding(7428, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [21]:
# Train LSTM
best_acc = 0

for epoch in range(epochs):
    lstm_model.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        lstm_optimizer.zero_grad()

        outputs = lstm_model(X)
        loss = lstm_criterion(outputs, y)

        loss.backward()
        lstm_optimizer.step()

        total_loss += loss.item()

    lstm_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = lstm_model(X)
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(lstm_model.state_dict(), "lstm_output_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 1.1036, Val Accuracy: 0.3600
Epoch 2/10, Loss: 1.0608, Val Accuracy: 0.3911
Epoch 3/10, Loss: 0.9449, Val Accuracy: 0.4756
Epoch 4/10, Loss: 0.7588, Val Accuracy: 0.5378
Epoch 5/10, Loss: 0.5873, Val Accuracy: 0.5511
Epoch 6/10, Loss: 0.4166, Val Accuracy: 0.5067
Epoch 7/10, Loss: 0.2428, Val Accuracy: 0.4800
Epoch 8/10, Loss: 0.1448, Val Accuracy: 0.5333
Epoch 9/10, Loss: 0.0788, Val Accuracy: 0.5111
Epoch 10/10, Loss: 0.0544, Val Accuracy: 0.5378


In [22]:
# Test LSTM
lstm_model.load_state_dict(torch.load(
    "lstm_output_best.pth",
    map_location=device,
    weights_only=True
))

lstm_model.eval()

y_true_lstm = []
y_pred_lstm = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)

        outputs = lstm_model(X)
        predictions = torch.argmax(outputs, dim=1)

        y_true_lstm.extend(y.tolist())
        y_pred_lstm.extend(predictions.cpu().tolist())

print("Test Accuracy:", accuracy_score(y_true_lstm, y_pred_lstm))
print("Macro F1:", f1_score(y_true_lstm, y_pred_lstm, average="macro"))

Test Accuracy: 0.5644444444444444
Macro F1: 0.5648765648765649


In [23]:
# Input only data
train_texts = train_df["LLM_Input"].astype(str).tolist()
val_texts = val_df["LLM_Input"].astype(str).tolist()
test_texts = test_df["LLM_Input"].astype(str).tolist()

word_count = Counter()

for text in train_texts:
    word_count.update(text.lower().split())

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, count in word_count.items():
    if count >= 2:
        vocab[word] = len(vocab)

print(len(vocab))

974


In [24]:
# Input only tensors
X_train = torch.tensor([encode(text) for text in train_texts], dtype=torch.long)
X_val = torch.tensor([encode(text) for text in val_texts], dtype=torch.long)
X_test = torch.tensor([encode(text) for text in test_texts], dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

print(X_train.shape)

torch.Size([1050, 200])


In [25]:
# CNN input only
cnn_input = CNN(len(vocab)).to(device)

cnn_optimizer = torch.optim.Adam(cnn_input.parameters(), lr=0.001)
cnn_criterion = nn.CrossEntropyLoss()

In [26]:
# Train CNN input
best_acc = -1

for epoch in range(epochs):
    cnn_input.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        cnn_optimizer.zero_grad()
        outputs = cnn_input(X)
        loss = cnn_criterion(outputs, y)
        loss.backward()
        cnn_optimizer.step()

        total_loss += loss.item()

    cnn_input.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            predictions = cnn_input(X).argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += len(y)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(cnn_input.state_dict(), "cnn_input_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 1.2516, Val Accuracy: 0.3333
Epoch 2/10, Loss: 1.2110, Val Accuracy: 0.3333
Epoch 3/10, Loss: 1.2116, Val Accuracy: 0.3333
Epoch 4/10, Loss: 1.1819, Val Accuracy: 0.3333
Epoch 5/10, Loss: 1.1672, Val Accuracy: 0.3333
Epoch 6/10, Loss: 1.1644, Val Accuracy: 0.3333
Epoch 7/10, Loss: 1.1456, Val Accuracy: 0.3333
Epoch 8/10, Loss: 1.1515, Val Accuracy: 0.3333
Epoch 9/10, Loss: 1.1400, Val Accuracy: 0.3333
Epoch 10/10, Loss: 1.1455, Val Accuracy: 0.3333


In [27]:
# Test CNN input
cnn_input.load_state_dict(torch.load(
    "cnn_input_best.pth", map_location=device, weights_only=True
))

cnn_input.eval()
true_cnn_input = []
pred_cnn_input = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        predictions = cnn_input(X).argmax(dim=1)

        true_cnn_input.extend(y.tolist())
        pred_cnn_input.extend(predictions.cpu().tolist())

cnn_input_acc = accuracy_score(true_cnn_input, pred_cnn_input)
cnn_input_f1 = f1_score(true_cnn_input, pred_cnn_input, average="macro")

print("Test Accuracy:", cnn_input_acc)
print("Macro F1:", cnn_input_f1)

Test Accuracy: 0.3333333333333333
Macro F1: 0.266107401700622


In [28]:
# LSTM input only
lstm_input = LSTM(len(vocab)).to(device)

lstm_optimizer = torch.optim.Adam(lstm_input.parameters(), lr=0.001)
lstm_criterion = nn.CrossEntropyLoss()

In [29]:
# Train LSTM input
best_acc = -1

for epoch in range(epochs):
    lstm_input.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        lstm_optimizer.zero_grad()
        outputs = lstm_input(X)
        loss = lstm_criterion(outputs, y)
        loss.backward()
        lstm_optimizer.step()

        total_loss += loss.item()

    lstm_input.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            predictions = lstm_input(X).argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += len(y)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(lstm_input.state_dict(), "lstm_input_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 1.1182, Val Accuracy: 0.3333
Epoch 2/10, Loss: 1.1121, Val Accuracy: 0.3333
Epoch 3/10, Loss: 1.1079, Val Accuracy: 0.3333
Epoch 4/10, Loss: 1.1077, Val Accuracy: 0.3333
Epoch 5/10, Loss: 1.1052, Val Accuracy: 0.3333
Epoch 6/10, Loss: 1.1073, Val Accuracy: 0.3333
Epoch 7/10, Loss: 1.1059, Val Accuracy: 0.3333
Epoch 8/10, Loss: 1.1054, Val Accuracy: 0.3333
Epoch 9/10, Loss: 1.1050, Val Accuracy: 0.3333
Epoch 10/10, Loss: 1.1071, Val Accuracy: 0.3333


In [30]:
# Test LSTM input
lstm_input.load_state_dict(torch.load(
    "lstm_input_best.pth", map_location=device, weights_only=True
))

lstm_input.eval()
true_lstm_input = []
pred_lstm_input = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        predictions = lstm_input(X).argmax(dim=1)

        true_lstm_input.extend(y.tolist())
        pred_lstm_input.extend(predictions.cpu().tolist())

lstm_input_acc = accuracy_score(true_lstm_input, pred_lstm_input)
lstm_input_f1 = f1_score(true_lstm_input, pred_lstm_input, average="macro")

print("Test Accuracy:", lstm_input_acc)
print("Macro F1:", lstm_input_f1)

Test Accuracy: 0.3333333333333333
Macro F1: 0.3296199924106901


In [31]:
# Input and output data
train_texts = (train_df["LLM_Input"].astype(str) + " <SEP> " + train_df["LLM_output"].astype(str)).tolist()
val_texts = (val_df["LLM_Input"].astype(str) + " <SEP> " + val_df["LLM_output"].astype(str)).tolist()
test_texts = (test_df["LLM_Input"].astype(str) + " <SEP> " + test_df["LLM_output"].astype(str)).tolist()

word_count = Counter()

for text in train_texts:
    word_count.update(text.lower().split())

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, count in word_count.items():
    if count >= 2:
        vocab[word] = len(vocab)

print(len(vocab))

7635


In [32]:
# Input and output tensors
X_train = torch.tensor([encode(text) for text in train_texts], dtype=torch.long)
X_val = torch.tensor([encode(text) for text in val_texts], dtype=torch.long)
X_test = torch.tensor([encode(text) for text in test_texts], dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

print(X_train.shape)

torch.Size([1050, 200])


In [33]:
# CNN input and output
cnn_both = CNN(len(vocab)).to(device)

cnn_optimizer = torch.optim.Adam(cnn_both.parameters(), lr=0.001)
cnn_criterion = nn.CrossEntropyLoss()

In [34]:
# Train CNN input and output
best_acc = -1

for epoch in range(epochs):
    cnn_both.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        cnn_optimizer.zero_grad()
        outputs = cnn_both(X)
        loss = cnn_criterion(outputs, y)
        loss.backward()
        cnn_optimizer.step()

        total_loss += loss.item()

    cnn_both.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            predictions = cnn_both(X).argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += len(y)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(cnn_both.state_dict(), "cnn_both_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 1.2214, Val Accuracy: 0.4178
Epoch 2/10, Loss: 0.9652, Val Accuracy: 0.5200
Epoch 3/10, Loss: 0.8510, Val Accuracy: 0.5600
Epoch 4/10, Loss: 0.7374, Val Accuracy: 0.6089
Epoch 5/10, Loss: 0.6520, Val Accuracy: 0.6222
Epoch 6/10, Loss: 0.5834, Val Accuracy: 0.6311
Epoch 7/10, Loss: 0.5006, Val Accuracy: 0.6489
Epoch 8/10, Loss: 0.4362, Val Accuracy: 0.6311
Epoch 9/10, Loss: 0.3714, Val Accuracy: 0.6178
Epoch 10/10, Loss: 0.3506, Val Accuracy: 0.6400


In [35]:
# Test CNN input and output
cnn_both.load_state_dict(torch.load(
    "cnn_both_best.pth", map_location=device, weights_only=True
))

cnn_both.eval()
true_cnn_both = []
pred_cnn_both = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        predictions = cnn_both(X).argmax(dim=1)

        true_cnn_both.extend(y.tolist())
        pred_cnn_both.extend(predictions.cpu().tolist())

cnn_both_acc = accuracy_score(true_cnn_both, pred_cnn_both)
cnn_both_f1 = f1_score(true_cnn_both, pred_cnn_both, average="macro")

print("Test Accuracy:", cnn_both_acc)
print("Macro F1:", cnn_both_f1)

Test Accuracy: 0.6711111111111111
Macro F1: 0.6739434354069784


In [36]:
# LSTM input and output
lstm_both = LSTM(len(vocab)).to(device)

lstm_optimizer = torch.optim.Adam(lstm_both.parameters(), lr=0.001)
lstm_criterion = nn.CrossEntropyLoss()

In [37]:
# Train LSTM input and output
best_acc = -1

for epoch in range(epochs):
    lstm_both.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        lstm_optimizer.zero_grad()
        outputs = lstm_both(X)
        loss = lstm_criterion(outputs, y)
        loss.backward()
        lstm_optimizer.step()

        total_loss += loss.item()

    lstm_both.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            predictions = lstm_both(X).argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += len(y)

    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(lstm_both.state_dict(), "lstm_both_best.pth")

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 1.1010, Val Accuracy: 0.3111
Epoch 2/10, Loss: 1.0537, Val Accuracy: 0.3378
Epoch 3/10, Loss: 0.9671, Val Accuracy: 0.4356
Epoch 4/10, Loss: 0.8226, Val Accuracy: 0.4000
Epoch 5/10, Loss: 0.7264, Val Accuracy: 0.5022
Epoch 6/10, Loss: 0.4897, Val Accuracy: 0.5422
Epoch 7/10, Loss: 0.2899, Val Accuracy: 0.5067
Epoch 8/10, Loss: 0.1578, Val Accuracy: 0.5289
Epoch 9/10, Loss: 0.0750, Val Accuracy: 0.5200
Epoch 10/10, Loss: 0.0625, Val Accuracy: 0.5467


In [38]:
# Test LSTM input and output
lstm_both.load_state_dict(torch.load(
    "lstm_both_best.pth", map_location=device, weights_only=True
))

lstm_both.eval()
true_lstm_both = []
pred_lstm_both = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        predictions = lstm_both(X).argmax(dim=1)

        true_lstm_both.extend(y.tolist())
        pred_lstm_both.extend(predictions.cpu().tolist())

lstm_both_acc = accuracy_score(true_lstm_both, pred_lstm_both)
lstm_both_f1 = f1_score(true_lstm_both, pred_lstm_both, average="macro")

print("Test Accuracy:", lstm_both_acc)
print("Macro F1:", lstm_both_f1)

Test Accuracy: 0.5422222222222223
Macro F1: 0.5404447441536454


In [39]:
# Save results
results = pd.DataFrame({
    "Model": ["CNN", "LSTM", "CNN", "LSTM", "CNN", "LSTM"],
    "Input": ["Output", "Output", "Input", "Input", "Input + Output", "Input + Output"],
    "Accuracy": [
        0.7466666666666667,
        0.5688888888888889,
        cnn_input_acc,
        lstm_input_acc,
        cnn_both_acc,
        lstm_both_acc
    ],
    "Macro F1": [
        0.7410502012607276,
        0.5742925144047163,
        cnn_input_f1,
        lstm_input_f1,
        cnn_both_f1,
        lstm_both_f1
    ]
})

print(results)
results.to_csv("results.csv", index=False)

  Model           Input  Accuracy  Macro F1
0   CNN          Output  0.746667  0.741050
1  LSTM          Output  0.568889  0.574293
2   CNN           Input  0.333333  0.266107
3  LSTM           Input  0.333333  0.329620
4   CNN  Input + Output  0.671111  0.673943
5  LSTM  Input + Output  0.542222  0.540445
